In [1]:
!pip install transformers torchaudio soundfile


In [2]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import soundfile as sf

c:\Users\zayna\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load model and processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Load and preprocess audio (mono, 16kHz)
def load_audio(file_path):
    audio_input, sample_rate = sf.read(file_path)
    if sample_rate != 16000:
        raise ValueError("Audio must be sampled at 16kHz")
    
    # Convert stereo to mono if needed
    if len(audio_input.shape) == 2:
        audio_input = audio_input.mean(axis=1)
    
    return audio_input


# Transcribe audio
def transcribe(audio_path):
    input_audio = load_audio(audio_path)
    inputs = processor(input_audio, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription.lower()

In [11]:
import torchaudio

# Load the audio
waveform, sample_rate = torchaudio.load("/Users/zayna/Downloads/twos.wav")

# Resample if needed
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
resampled_waveform = resampler(waveform)

# Save the resampled audio
torchaudio.save("/Users/zayna/Downloads/twos_16k.wav", resampled_waveform, 16000)

In [12]:
audio_path = "/Users/zayna/Downloads/twos_16k.wav"
text = transcribe(audio_path)
print(text)


he jimmie are you hungry i'm quite hungry what shall we cook oh my boy alex i have this recipic idea ayo done yea tell me the this chickan be anne um that is spicy arononificateeterit dispessionis yea we fine i think i can go on so you you may let the chicken add enough chilli paoda and let it sit for twenty minutes oh whell that sounds yummy yes after i plan fry it and mix it with vrice look hee look here ou out outrighter let le's do that then yea please le menhavi go i will thank you


In [7]:
!pip install datasets

In [8]:
!pip install evaluate

In [9]:
!pip install jiwer

In [13]:
import evaluate

# Load WER and CER metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Your ground truth transcription (from image)
reference = """
hey jamie are you hungry what should we cook oh my god i have this recipe idea are you down
yeah tell me this chicken biryani that is spicy i don't know if you can tolerate
yeah i'll be fine i think i can go on so you marinate the chicken add enough chili powder and let it sit for 20 minutes
oh wow that sounds yummy yes after you pan fry it and mix it with rice okay okay i'll try that
let's do that then yeah please let me know how it goes i will thank you
"""

# Model prediction (your output from earlier)
prediction = """
he jimmie are you hungry i'm quite hungry what shall we cook oh my boy alex i have this recipic idea ayo done yea tell me the this chickan be anne um that is spicy arononificateeterit dispessionis yea we fine i think i can go on so you you may let the chicken add enough chilli paoda and let it sit for twenty minutes oh whell that sounds yummy yes after i plan fry it and mix it with vrice look hee look here ou out outrighter let le's do that then yea please le menhavi go i will thank you
"""

# Normalize and flatten text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute and display metrics
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER (Word Error Rate): {wer:.2%}")
print(f"CER (Character Error Rate): {cer:.2%}")


WER (Word Error Rate): 54.55%
CER (Character Error Rate): 35.08%
